In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

##################################
# GARCH-X，照計畫書的公式，但效果很差， VAR的 STD 是用 GARCH的SHAPE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [2]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"


Agent pid 10711
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-9ee1b2f
# github.com:22 SSH-2.0-9ee1b2f
# github.com:22 SSH-2.0-9ee1b2f
# github.com:22 SSH-2.0-9ee1b2f
# github.com:22 SSH-2.0-9ee1b2f
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [3]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
python3-dev is already the newest

In [4]:
# =============================================================================
# ES1 GK-LSTM VaR-t  ── B 版本 (Walk-Forward Warm Update)
# =============================================================================
# 研究設計：
#   Stage 1 ── Base Model 訓練（2006-2021, batch training）
#   Stage 2 ── Walk-Forward 評估（2022-2025）
#               每日：predict → record → warm update → slide
#
# 目標變數 Y : gk_vol_daily  (GK 單日波動度，σ 尺度，不需再開根號)
# 輸入特徵 X : ES1_LN_RET, gk_vol_daily, garch_vol, VIX_CLOSE  (lookback=20)
# VaR      : t 分配, μ=0, shape clip(6,10), α=0.05 / 0.01
# Backtest : Kupiec UC + Christoffersen CC  (訓練期 & 滾動評估期)
# Baseline : GARCH 模型 (來自 es1_volatility_all_methods.csv)
# =============================================================================

import os, copy, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import t as tdist, chi2, pearsonr

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
warnings.filterwarnings('ignore')

# =============================================================================
# 0.  設定區  ── 所有參數集中在此，修改這裡即可
# =============================================================================

# ── 資料路徑 ──────────────────────────────────────────────────────────────────
PATH_DAY  = "./filtered_output/df_day.csv"
PATH_VOL  = "./real_volatility_multi_var/es1_volatility_all_methods.csv"

# ── 樣本切分 ──────────────────────────────────────────────────────────────────
TRAIN_START = "2006-01-01"
TRAIN_END   = "2021-12-31"
TEST_START  = "2022-01-01"


# ── 特徵 & 目標 ───────────────────────────────────────────────────────────────
FEATURE_COLS = ['ES1_LN_RET', 'gk_vol_daily', 'garch_vol', 'VIX_CLOSE']
TARGET_COL   = 'gk_vol_daily'
LOOKBACK     = 20         # 每次拿最近 20 天資料，預測第 21 天


# ── GPU 設定（有 GPU 自動加速；無 GPU 正常跑 CPU）────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("ℹ  No GPU, running on CPU")


# =============================================================================
# 1.  讀資料 & 建母資料表
# =============================================================================

def load_master(path_day: str, path_vol: str) -> pd.DataFrame:
    """
    合併 df_day.csv 與 es1_volatility_all_methods.csv，
    以 DATE 對齊，篩選 2006 年之後，刪除特徵缺值。
    """
    # ── df_day ──
    day = pd.read_csv(path_day)
    if 'DATE' not in day.columns:
        day = day.reset_index().rename(columns={day.columns[0]: 'DATE'})
    day['DATE'] = pd.to_datetime(day['DATE'], errors='coerce').dt.normalize()

    keep_day = ['DATE', 'ES1_LN_RET', 'ES1_CLOSE', 'ES1_VOLUME',
                'VIX_LN_RET', 'VIX_CLOSE']
    day = day[[c for c in keep_day if c in day.columns]].dropna(subset=['DATE'])

    # ── vol / garch / shape ──
    vol = pd.read_csv(path_vol)
    if 'DATE' not in vol.columns:
        vol = vol.reset_index().rename(columns={vol.columns[0]: 'DATE'})
    vol['DATE'] = pd.to_datetime(vol['DATE'], errors='coerce').dt.normalize()

    keep_vol = ['DATE', 'gk_vol_daily', 'garch_vol', 'shape']
    vol = vol[[c for c in keep_vol if c in vol.columns]].dropna(subset=['DATE'])

    # ── merge ──
    df = pd.merge(day, vol, on='DATE', how='inner')
    df = df.sort_values('DATE').reset_index(drop=True)
    df = df[df['DATE'] >= TRAIN_START].reset_index(drop=True)
    df = df.dropna(subset=FEATURE_COLS + ['shape']).reset_index(drop=True)

    print(f"Master table: {len(df):,} rows  "
          f"({df['DATE'].min().date()} ~ {df['DATE'].max().date()})")
    return df


df = load_master(PATH_DAY, PATH_VOL)

for c in df.columns:
    print(c)


ℹ  No GPU, running on CPU
Master table: 4,922 rows  (2006-01-03 ~ 2025-06-30)
DATE
ES1_LN_RET
ES1_CLOSE
ES1_VOLUME
VIX_LN_RET
VIX_CLOSE
gk_vol_daily
garch_vol
shape


In [5]:
# =============================================================================
# 2.  特徵縮放 & 滑動視窗序列建構
# =============================================================================
# ── Base Model 超參數 ─────────────────────────────────────────────────────────
LSTM_UNITS   = 64               # LSTM 隱藏層神經元數
DROPOUT      = 0.2              # Dropout 比率
LR_BASE      = 1e-3             # Base Model 學習率
EPOCHS       = 100              # 最大 epoch（配合 EarlyStopping）
BATCH_SIZE   = 32
PATIENCE_ES  = 20               # EarlyStopping patience
PATIENCE_LR  = 10               # ReduceLROnPlateau patience

# ── Walk-Forward Warm Update 超參數 ──────────────────────────────────────────
LR_WARM      = 1e-4             # Warm update 低學習率（比 Base LR 低 10x）
WARM_EPOCHS  = 1                # 每次更新 1 epoch（避免單日噪音過擬合）
WARM_BATCH   = 1                # 單樣本更新

# ── VaR 設定 ──────────────────────────────────────────────────────────────────
ALPHA_95     = 0.05
ALPHA_99     = 0.01
NU_CLIP      = (6, 10)          # shape 截尾範圍

OUTPUT_DIR = "./LSTM_B_diagnostics"
os.makedirs(OUTPUT_DIR, exist_ok=True)




train_mask = (df['DATE'] >= TRAIN_START) & (df['DATE'] <= TRAIN_END)
test_mask  = df['DATE'] >= TEST_START

# ── Scaler：只用訓練集 fit，避免 data leakage ─────────────────────────────────
#縮放，但我覺得這可以討論
scaler_X = MinMaxScaler()
scaler_X.fit(df.loc[train_mask, FEATURE_COLS])
X_scaled_all = scaler_X.transform(df[FEATURE_COLS].values)   # 全樣本縮放

scaler_y = MinMaxScaler()
scaler_y.fit(df.loc[train_mask, [TARGET_COL]])
Y_scaled_all = scaler_y.transform(
    df[[TARGET_COL]].values).ravel()                          # 全樣本縮放

def descale_vol(y_scaled: np.ndarray) -> np.ndarray:
    """反縮放並確保正值"""
    return np.clip(
        scaler_y.inverse_transform(
            np.array(y_scaled).reshape(-1, 1)).ravel(),
        1e-8, None)

# ── 滑動視窗序列 ──────────────────────────────────────────────────────────────
def build_sequences(X_scaled, Y_scaled, dates, shapes, returns, true_vol,
                    lookback=LOOKBACK):
    # """
    # X[i] = X[i-lookback : i]    (過去 lookback 天的特徵，shape=(lookback, n_feat))
    # Y[i] = Y[i]                 (第 i 天的目標值，已縮放)

    # 滑動規則（與 PDF 定義一致）：
    #   第 1 筆：用第 1~20 天特徵 → 預測第 21 天
    #   第 2 筆：用第 2~21 天特徵 → 預測第 22 天  ...
    # """
    Xs, Ys = [], []
    seq_dates, seq_shapes, seq_ret, seq_vol = [], [], [], []

    for i in range(lookback, len(X_scaled)):
        # 取出「前 20 天的 X」，當成一筆輸入序列
        Xs.append(X_scaled[i - lookback: i])   # (lookback, n_feat)
        Ys.append(Y_scaled[i])
        seq_dates.append(dates[i])
        seq_shapes.append(shapes[i])
        seq_ret.append(returns[i])
        seq_vol.append(true_vol[i])

    return (np.array(Xs, dtype=np.float32),
            np.array(Ys, dtype=np.float32),
            np.array(seq_dates),
            np.array(seq_shapes, dtype=np.float32),
            np.array(seq_ret,    dtype=np.float32),
            np.array(seq_vol,    dtype=np.float32))


(X_seq, Y_seq, seq_dates,
 seq_shapes, seq_ret, seq_vol) = build_sequences(
    X_scaled_all, Y_scaled_all,
    df['DATE'].values,
    df['shape'].values,
    df['ES1_LN_RET'].values,
    df[TARGET_COL].values
)

# ── 訓練 / 測試切分 ───────────────────────────────────────────────────────────
# 目標日期在 2022/01/01 之前 → train
# 目標日期在 2022/01/01 之後 → test
tr_mask = pd.to_datetime(seq_dates) <= pd.Timestamp(TRAIN_END)
te_mask = pd.to_datetime(seq_dates) >= pd.Timestamp(TEST_START)

X_train, Y_train = X_seq[tr_mask], Y_seq[tr_mask]
X_test,  Y_test  = X_seq[te_mask], Y_seq[te_mask]

dates_tr  = pd.to_datetime(seq_dates[tr_mask])
dates_te  = pd.to_datetime(seq_dates[te_mask])
shape_tr  = seq_shapes[tr_mask];  shape_te  = seq_shapes[te_mask]
ret_tr    = seq_ret[tr_mask];     ret_te    = seq_ret[te_mask]
true_vol_tr = seq_vol[tr_mask];   true_vol_te = seq_vol[te_mask]

print(f"Train: {X_train.shape}  ({dates_tr[0].date()} ~ {dates_tr[-1].date()})")
print(f"Test : {X_test.shape}   ({dates_te[0].date()} ~ {dates_te[-1].date()})")


# =============================================================================
# 3.  建立 LSTM 模型（tensorflow.keras）
# =============================================================================

def build_lstm_model(lookback: int,
                     n_features: int,
                     lstm_units: int = LSTM_UNITS,
                     dropout: float  = DROPOUT,
                     lr: float       = LR_BASE) -> tf.keras.Model:
    # """
    # 架構：Input → LSTM → Dropout → Dense(softplus)
    # softplus 輸出恆正，與 volatility 的非負特性一致。
    # 損失函數使用 Huber（delta=1），對極端值比純 MSE 更穩健。
    # """
    inp = layers.Input(shape=(lookback, n_features), name="seq_input")
    x   = layers.LSTM(lstm_units, return_sequences=False, name="lstm_1")(inp)
    x   = layers.Dropout(dropout, name="dropout")(x)
    # 一個 Dense 輸出層，使用 softplus 保證輸出為正值
    out = layers.Dense(1, activation='softplus', name="vol_output")(x)

    model = models.Model(inp, out, name="GK_LSTM")
    model.compile(
        # 金融資料常有極端值，Huber 比單純 MSE 對極端值穩健一些
        # 可嘗試 loss='mse'，Huber → MSE → Weighted MSE / Quantile Loss
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.Huber(delta=1.0)   # Huber loss
    )
    return model


n_features = X_train.shape[2]     # = 4
print(f"\nModel input shape: ({LOOKBACK}, {n_features})")


# =============================================================================
# 4.  Stage 1：Base Model 訓練（2006–2021, batch training）
# =============================================================================

print("\n" + "="*60)
print("STAGE 1: Base Model Training (2006-2021)")
print("="*60)

#建立模型的「外形」，不是訓練
base_model = build_lstm_model(LOOKBACK, n_features, lr=LR_BASE)
base_model.summary()

cb_list = [
    # EarlyStopping：驗證 loss 不改善 PATIENCE_ES 個 epoch 就停止，並還原最佳權重
    callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE_ES,
        restore_best_weights=True, verbose=1),
    # ReduceLROnPlateau：驗證 loss 停滯時自動降低學習率
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=PATIENCE_LR, min_lr=1e-6, verbose=1),
    # ModelCheckpoint：儲存最佳模型
    callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "base_model_best.keras"),
        monitor='val_loss', save_best_only=True, verbose=0),
]

#把 X_train 丟進去，讓模型預測，再拿 Y_train 告訴它正確答案是什麼，然後修正權重
history = base_model.fit(
    X_train, Y_train,
    validation_split=0.1,     # 訓練集末 10% 為驗證集（時序保留，不打亂）
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,            # 時間序列嚴禁打亂
    callbacks=cb_list,
    verbose=1
)

print(f"\n✓ Base model trained: {len(history.history['loss'])} epochs")

# ── In-sample 預測（訓練期）────────────────────────────────────────────────────
# Base Model 在訓練集裡學得怎麼樣
pred_tr_scaled = base_model.predict(X_train, verbose=0).ravel()
pred_tr = descale_vol(pred_tr_scaled)    # 還原至 vol 尺度

rmse_tr = float(np.sqrt(mean_squared_error(true_vol_tr, pred_tr)))
mae_tr  = float(mean_absolute_error(true_vol_tr, pred_tr))
corr_tr = float(pearsonr(pred_tr, true_vol_tr)[0])
print(f"[Stage 1 Train] RMSE={rmse_tr:.6f}  MAE={mae_tr:.6f}  Corr={corr_tr:.4f}")



Train: (4025, 20, 4)  (2006-01-31 ~ 2021-12-31)
Test : (877, 20, 4)   (2022-01-03 ~ 2025-06-30)

Model input shape: (20, 4)

STAGE 1: Base Model Training (2006-2021)


Model: "GK_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ seq_input (InputLayer)          │ (None, 20, 4)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vol_output (Dense)              │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,729 (69.25 KB)

 Trainable params: 17,729 (69.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0211 - val_loss: 0.0021 - learning_rate: 0.0010
Epoch 2/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0047 - val_loss: 0.0020 - learning_rate: 0.0010
Epoch 3/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0044 - val_loss: 0.0018 - learning_rate: 0.0010
Epoch 4/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0037 - val_loss: 0.0016 - learning_rate: 0.0010
Epoch 5/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0027 - val_loss: 0.0017 - learning_rate: 0.0010
Epoch 6/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0027 - val_loss: 0.0017 - learning_rate: 0.0010
Epoch 7/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0026 - val_loss: 0.0018 - learning_rate: 0.0010
Epoch 8/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0024 - val_loss: 0.0017 - learning_rate: 0.0010
Epoch 9/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0023 - val_loss: 0.0018 - learning_rate: 0.00

In [6]:
# ── Frozen model 測試集預測（作為 A版基準對照）────────────────────────────────
# 訓練完後，把模型固定住，不再更新，直接拿去預測 2022–2025。
pred_frozen_scaled = base_model.predict(X_test, verbose=0).ravel()
pred_frozen = descale_vol(pred_frozen_scaled)

rmse_frozen = float(np.sqrt(mean_squared_error(true_vol_te, pred_frozen)))
mae_frozen  = float(mean_absolute_error(true_vol_te, pred_frozen))
corr_frozen = float(pearsonr(pred_frozen, true_vol_te)[0])
print(f"[Frozen A-ver ] RMSE={rmse_frozen:.6f}  MAE={mae_frozen:.6f}  Corr={corr_frozen:.4f}")

# ── 保存 Base Model 完整權重（作為後續 warm update 起點）────────────────────────
base_model.save(os.path.join(OUTPUT_DIR, "base_model_final.keras"))
print(f"✓ Base model saved.")


# =============================================================================
# 5.  Stage 2：Walk-Forward Warm Update（2022–2025）
# =============================================================================
# 每日流程：
#   Step A: predict（使用 [t-20, t-1] 特徵，在看到 t 日真值前）
#   Step B: record（記錄預測值，作為 VaR 建構依據）
#   Step C: warm update（以低學習率、1 epoch 更新模型，不重置權重）
#   Step D: slide（視窗前移一日，繼續下一天）
# =============================================================================

print("\n" + "="*60)
print(f"STAGE 2: Walk-Forward Warm Update (2022-2025)")
print(f"  Warm LR={LR_WARM}, Warm Epochs={WARM_EPOCHS}, Warm Batch={WARM_BATCH}")
print("="*60)

# ── 建立 Warm Model：複製 base model 權重，降低學習率 ─────────────────────────
# 方法：重建相同架構，載入 base model 的最佳權重，重新編譯為低 LR
warm_model = build_lstm_model(LOOKBACK, n_features, lr=LR_WARM)
warm_model.set_weights(base_model.get_weights())   # 完整複製權重

# 驗證複製是否成功
check_base = base_model.predict(X_test[:3], verbose=0).ravel()
check_warm = warm_model.predict(X_test[:3], verbose=0).ravel()
assert np.allclose(check_base, check_warm, atol=1e-5), "Weight copy failed!"
print("✓ Warm model weights copied from base model.")

# ── 取得測試期間在全樣本中的位置 ──────────────────────────────────────────────
full_dates   = df['DATE'].values
test_pos_start = np.searchsorted(full_dates, np.datetime64(TEST_START))

# 滾動評估結果容器
walk_pred   = []    # 每日 one-step-ahead 預測（更新前記錄）
walk_dates  = []
walk_shapes = []
walk_ret    = []
walk_vol    = []    # 真實 gk_vol_daily

n_walk = len(full_dates) - test_pos_start
print(f"Walk-forward window: {n_walk} days")

# WARM
# 預測必須在更新前完成
# 預測值要先記錄下來
# 後續 VaR 也要用更新前的預測值去建
for i, pos in enumerate(range(test_pos_start, len(df))):
    if pos < LOOKBACK:
        continue

    # ── 建立輸入序列 ──────────────────────────────────────────────────────────
    window = X_scaled_all[pos - LOOKBACK: pos]      # shape: (20, 4)
    x_input = window.reshape(1, LOOKBACK, n_features).astype(np.float32)

    # ── Step A & B: 預測 + 記錄（必須在更新前完成）────────────────────────────
    y_pred_scaled = warm_model.predict(x_input, verbose=0).ravel()[0]
    y_pred_vol    = float(descale_vol(np.array([y_pred_scaled]))[0])

    walk_pred.append(y_pred_vol)
    walk_dates.append(full_dates[pos])
    walk_shapes.append(float(df['shape'].values[pos]))
    walk_ret.append(float(df['ES1_LN_RET'].values[pos]))
    walk_vol.append(float(df[TARGET_COL].values[pos]))

    # ── Step C: Warm Update（用真實 y 更新，低學習率，1 epoch）────────────────
    y_true_scaled = float(Y_scaled_all[pos])
    warm_model.fit(
        x_input,
        np.array([[y_true_scaled]], dtype=np.float32),
        epochs=WARM_EPOCHS,
        batch_size=WARM_BATCH,
        verbose=0,
        shuffle=False
    )

    # ── 進度顯示 ──────────────────────────────────────────────────────────────
    if (i + 1) % 200 == 0 or (i + 1) == n_walk:
        print(f"  Walk-forward: {i+1:4d}/{n_walk} days done")

# ── 整理 walk-forward 結果 ────────────────────────────────────────────────────
walk_pred   = np.array(walk_pred,   dtype=np.float32)
walk_dates  = pd.to_datetime(walk_dates)
walk_shapes = np.array(walk_shapes, dtype=np.float32)
walk_ret    = np.array(walk_ret,    dtype=np.float32)
walk_vol    = np.array(walk_vol,    dtype=np.float32)

rmse_wf = float(np.sqrt(mean_squared_error(walk_vol, walk_pred)))
mae_wf  = float(mean_absolute_error(walk_vol, walk_pred))
corr_wf = float(pearsonr(walk_pred, walk_vol)[0])
print(f"\n[Walk-Forward B-ver] RMSE={rmse_wf:.6f}  MAE={mae_wf:.6f}  Corr={corr_wf:.4f}")
print(f"[Frozen    A-ver   ] RMSE={rmse_frozen:.6f}  MAE={mae_frozen:.6f}  Corr={corr_frozen:.4f}")
print(f"RMSE change: {(rmse_wf-rmse_frozen)/rmse_frozen*100:+.1f}%  "
      f"MAE change: {(mae_wf-mae_frozen)/mae_frozen*100:+.1f}%")

# 保存 warm model 最終狀態
warm_model.save(os.path.join(OUTPUT_DIR, "warm_model_final.keras"))
print(f"✓ Warm model saved.")


# =============================================================================
# 6.  VaR-t 建構
# =============================================================================

def build_var_t(sigma: np.ndarray, shape: np.ndarray,
                alpha: float = ALPHA_95) -> np.ndarray:
    # """
    # VaR-t 公式：
    #   ν = clip(shape, 6, 10)
    #   s = sqrt((ν-2)/ν)          # Var=1 標準化修正
    #   VaR = σ × s × t_ν^{-1}(α)  # 負值（左尾損失）
    # """
    nu = np.clip(shape, *NU_CLIP)
    s  = np.sqrt((nu - 2.0) / nu)
    q  = tdist.ppf(alpha, df=nu)   # 分位數，< 0
    return sigma * s * q            # 負值


# ── 訓練期 VaR ────────────────────────────────────────────────────────────────
var95_tr = build_var_t(pred_tr,    shape_tr, ALPHA_95)
var99_tr = build_var_t(pred_tr,    shape_tr, ALPHA_99)

# ── Frozen 測試期 VaR（A版基準）────────────────────────────────────────────────
var95_frozen = build_var_t(pred_frozen, shape_te, ALPHA_95)
var99_frozen = build_var_t(pred_frozen, shape_te, ALPHA_99)

# ── Walk-Forward 測試期 VaR（B版，使用更新前預測值）──────────────────────────
var95_wf = build_var_t(walk_pred, walk_shapes, ALPHA_95)
var99_wf = build_var_t(walk_pred, walk_shapes, ALPHA_99)

# ── 違規判定 ──────────────────────────────────────────────────────────────────
viol95_tr     = (ret_tr < var95_tr).astype(int)
viol99_tr     = (ret_tr < var99_tr).astype(int)
viol95_frozen = (ret_te < var95_frozen).astype(int)
viol99_frozen = (ret_te < var99_frozen).astype(int)
viol95_wf     = (walk_ret < var95_wf).astype(int)
viol99_wf     = (walk_ret < var99_wf).astype(int)

# ── GARCH baseline VaR（從 es1_volatility_all_methods.csv 取 garch_vol）──────
vol_csv = pd.read_csv(PATH_VOL, parse_dates=['DATE'])
vol_csv['DATE'] = pd.to_datetime(vol_csv['DATE'], errors='coerce').dt.normalize()
garch_map = vol_csv.set_index('DATE')['garch_vol']
shape_map = vol_csv.set_index('DATE')['shape']

garch_te  = np.array([garch_map.get(d, np.nan) for d in walk_dates], dtype=np.float32)
shape_garch = np.array([shape_map.get(d, np.nan) for d in walk_dates], dtype=np.float32)
valid_g = ~np.isnan(garch_te)

var95_garch  = np.full(len(walk_ret), np.nan)
var99_garch  = np.full(len(walk_ret), np.nan)
var95_garch[valid_g] = build_var_t(garch_te[valid_g], shape_garch[valid_g], ALPHA_95)
var99_garch[valid_g] = build_var_t(garch_te[valid_g], shape_garch[valid_g], ALPHA_99)

viol95_garch = (walk_ret[valid_g] < var95_garch[valid_g]).astype(int)

# ── GK Realized 基準（同日實際 GK vol，僅供診斷參考）─────────────────────────
gk_map = vol_csv.set_index('DATE')['gk_vol_daily']
gk_te  = np.array([gk_map.get(d, np.nan) for d in walk_dates], dtype=np.float32)
valid_gk = ~np.isnan(gk_te)
var95_gk_real = build_var_t(gk_te[valid_gk], shape_garch[valid_gk], ALPHA_95)
viol95_gk     = (walk_ret[valid_gk] < var95_gk_real).astype(int)


# =============================================================================
# 7.  回測統計：Kupiec UC + Christoffersen CC
# =============================================================================

def kupiec_test(viol: np.ndarray, alpha: float) -> dict:
    """Kupiec (1995) POF / Unconditional Coverage Test"""
    v = np.asarray(viol, dtype=int)
    n = len(v);  x = int(v.sum());  eps = 1e-12
    phat = np.clip(x / n, eps, 1 - eps)
    ac   = np.clip(alpha, eps, 1 - eps)
    ll_h0  = (n-x)*np.log(1-ac)   + x*np.log(ac)
    ll_mle = (n-x)*np.log(1-phat) + x*np.log(phat)
    LR_uc = -2.0 * (ll_h0 - ll_mle)
    p_val = 1.0 - chi2.cdf(LR_uc, df=1)
    return dict(alpha=alpha, n=n, x=x, viol_rate=float(x/n),
                LR_uc=float(LR_uc), p_uc=float(p_val))


def christoffersen_cc_test(viol: np.ndarray, alpha: float) -> dict:
    """Christoffersen (1998) Conditional Coverage Test (UC + IND)"""
    v = np.asarray(viol, dtype=int)
    n = len(v)
    uc = kupiec_test(v, alpha)

    v_lag, v_now = v[:-1], v[1:]
    n00 = int(np.sum((v_lag==0) & (v_now==0)))
    n01 = int(np.sum((v_lag==0) & (v_now==1)))
    n10 = int(np.sum((v_lag==1) & (v_now==0)))
    n11 = int(np.sum((v_lag==1) & (v_now==1)))

    eps   = 1e-12
    pi01  = np.clip(n01 / max(n00+n01, 1), eps, 1-eps)
    pi11  = np.clip(n11 / max(n10+n11, 1), eps, 1-eps)
    pi    = np.clip((n01+n11) / max(n00+n01+n10+n11, 1), eps, 1-eps)

    ll_h0_ind = (n00+n10)*np.log(1-pi)  + (n01+n11)*np.log(pi)
    ll_h1_ind = (n00*np.log(1-pi01) + n01*np.log(pi01) +
                 n10*np.log(1-pi11) + n11*np.log(pi11))

    LR_ind = -2.0 * (ll_h0_ind - ll_h1_ind)
    p_ind  = 1.0 - chi2.cdf(LR_ind, df=1)
    LR_cc  = uc['LR_uc'] + LR_ind
    p_cc   = 1.0 - chi2.cdf(LR_cc, df=2)

    return dict(
        alpha=alpha, n=n,
        x=uc['x'], viol_rate=uc['viol_rate'],
        n00=n00, n01=n01, n10=n10, n11=n11,
        LR_uc=uc['LR_uc'],   p_uc=uc['p_uc'],
        LR_ind=float(LR_ind), p_ind=float(p_ind),
        LR_cc=float(LR_cc),   p_cc=float(p_cc),
    )


def print_backtest(viol: np.ndarray, alpha: float, label: str):
    k = kupiec_test(viol, alpha)
    c = christoffersen_cc_test(viol, alpha)
    print(f"  [{label}]  N={k['n']}, Viol={k['x']} ({k['viol_rate']*100:.2f}%)  "
          f"UC p={k['p_uc']:.4f} {'✓' if k['p_uc']>=0.05 else '✗'}  "
          f"CC p={c['p_cc']:.4f} {'✓' if c['p_cc']>=0.05 else '✗'}  "
          f"IND p={c['p_ind']:.4f}  "
          f"n00/01/10/11={c['n00']}/{c['n01']}/{c['n10']}/{c['n11']}")
    return k, c


print("\n" + "="*70)
print("VaR(95%) Backtest Results")
print("="*70)
k95_tr,   c95_tr   = print_backtest(viol95_tr,     ALPHA_95, "Base Model Train 2006-2021")
k95_frz,  c95_frz  = print_backtest(viol95_frozen,  ALPHA_95, "Frozen  A-ver  Test 2022-2025")
k95_wf,   c95_wf   = print_backtest(viol95_wf,      ALPHA_95, "Walk-Fwd B-ver Test 2022-2025")
k95_g,    c95_g    = print_backtest(viol95_garch,   ALPHA_95, "GARCH Baseline  Test 2022-2025")
k95_gkr,  c95_gkr  = print_backtest(viol95_gk,      ALPHA_95, "GK Realized     Test 2022-2025 *")

print("\n" + "="*70)
print("VaR(99%) Backtest Results")
print("="*70)
k99_tr,   c99_tr   = print_backtest(viol99_tr,     ALPHA_99, "Base Model Train 2006-2021")
k99_frz,  c99_frz  = print_backtest(viol99_frozen,  ALPHA_99, "Frozen  A-ver  Test 2022-2025")
k99_wf,   c99_wf   = print_backtest(viol99_wf,      ALPHA_99, "Walk-Fwd B-ver Test 2022-2025")


# =============================================================================
# 8.  整理輸出表格（CSV）
# =============================================================================

def make_summary_row(k: dict, c: dict, split: str, model: str, level: str) -> dict:
    return {
        'model': model, 'split': split, 'level': level,
        'N': k['n'], 'violations': k['x'],
        'viol_rate': round(k['viol_rate'], 6),
        'kupiec_LR': round(k['LR_uc'], 4), 'kupiec_p': round(k['p_uc'], 4),
        'cc_LR': round(c['LR_cc'], 4),     'cc_p': round(c['p_cc'], 4),
        'ind_p': round(c['p_ind'], 4),
        'n00': c['n00'], 'n01': c['n01'], 'n10': c['n10'], 'n11': c['n11'],
        'uc_pass': k['p_uc'] >= 0.05,
        'cc_pass': c['p_cc'] >= 0.05,
    }


df_backtest = pd.DataFrame([
    make_summary_row(k95_tr,  c95_tr,  'train',      'LSTM_Base',    '95'),
    make_summary_row(k99_tr,  c99_tr,  'train',      'LSTM_Base',    '99'),
    make_summary_row(k95_frz, c95_frz, 'test_frozen','LSTM_Frozen',  '95'),
    make_summary_row(k99_frz, c99_frz, 'test_frozen','LSTM_Frozen',  '99'),
    make_summary_row(k95_wf,  c95_wf,  'test_wf',    'LSTM_WF_B',    '95'),
    make_summary_row(k99_wf,  c99_wf,  'test_wf',    'LSTM_WF_B',    '99'),
    make_summary_row(k95_g,   c95_g,   'test_wf',    'GARCH',        '95'),
    make_summary_row(k95_gkr, c95_gkr, 'test_wf',    'GK_Realized',  '95'),
])
bt_path = os.path.join(OUTPUT_DIR, "backtest_summary.csv")
df_backtest.to_csv(bt_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Saved: {bt_path}")

# 波動預測指標
df_vol_metrics = pd.DataFrame([
    {'model': 'LSTM_Base_Train',  'split':'train',  'RMSE': rmse_tr,     'MAE': mae_tr,     'Corr': corr_tr},
    {'model': 'LSTM_Frozen_Test', 'split':'frozen', 'RMSE': rmse_frozen, 'MAE': mae_frozen, 'Corr': corr_frozen},
    {'model': 'LSTM_WF_B_Test',   'split':'wf',     'RMSE': rmse_wf,     'MAE': mae_wf,     'Corr': corr_wf},
])
vm_path = os.path.join(OUTPUT_DIR, "vol_forecast_metrics.csv")
df_vol_metrics.to_csv(vm_path, index=False, encoding='utf-8-sig')
print(f"✓ Saved: {vm_path}")

# 逐日匯出（Walk-Forward 測試集）
df_daily_wf = pd.DataFrame({
    'DATE':          walk_dates,
    'ret_true':      walk_ret,
    'sigma_lstm_wf': walk_pred,
    'gk_vol_actual': walk_vol,
    'shape':         walk_shapes,
    'VaR_ret_95':    var95_wf,
    'VaR_ret_99':    var99_wf,
    'viol_95':       viol95_wf,
    'viol_99':       viol99_wf,
}).set_index('DATE')
df_daily_wf.to_csv(os.path.join(OUTPUT_DIR, "var_daily_wf.csv"), encoding='utf-8-sig')

# 逐日匯出（Base Model 訓練期）
df_daily_tr = pd.DataFrame({
    'DATE':          dates_tr,
    'ret_true':      ret_tr,
    'sigma_lstm':    pred_tr,
    'gk_vol_actual': true_vol_tr,
    'shape':         shape_tr,
    'VaR_ret_95':    var95_tr,
    'VaR_ret_99':    var99_tr,
    'viol_95':       viol95_tr,
    'viol_99':       viol99_tr,
}).set_index('DATE')
df_daily_tr.to_csv(os.path.join(OUTPUT_DIR, "var_daily_train.csv"), encoding='utf-8-sig')
print(f"✓ Saved daily CSV files.")


# =============================================================================
# 9.  圖表輸出
# =============================================================================

# ── Fig 1: 2×3 Master Overview ───────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle('ES1 GK-LSTM B Version: Base Model + Walk-Forward Warm Update',
             fontsize=14, fontweight='bold')
axes = fig.subplots(2, 3)

# Row 0: Vol forecast
for ax, dates, pred, true, label, rmse, mae, corr in [
    (axes[0,0], dates_tr, pred_tr,    true_vol_tr,
     'Base Model Vol (Train 2006-2021)',
     rmse_tr, mae_tr, corr_tr),
    (axes[0,1], walk_dates, pred_frozen, true_vol_te,
     'Frozen Vol (Test 2022-2025)',
     rmse_frozen, mae_frozen, corr_frozen),
    (axes[0,2], walk_dates, walk_pred, walk_vol,
     'Walk-Forward Vol (Test 2022-2025)',
     rmse_wf, mae_wf, corr_wf),
]:
    ax.plot(dates, true, color='#333', lw=0.6, alpha=0.9, label='Actual')
    ax.plot(dates, pred, color='#1565C0', lw=0.7, alpha=0.85, label='Predicted')
    ax.set_title(f'{label}\nRMSE={rmse:.5f} MAE={mae:.5f} Corr={corr:.3f}', fontsize=9)
    ax.set_ylabel('σ'); ax.legend(fontsize=7); ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))

# Row 1: Return vs VaR
for ax, dates, ret, var, viol, label in [
    (axes[1,0], dates_tr, ret_tr, var95_tr, viol95_tr,
     f"VaR95 Train  Viol={viol95_tr.sum()} ({viol95_tr.mean()*100:.1f}%)"),
    (axes[1,1], walk_dates, walk_ret, var95_frozen, viol95_frozen,
     f"VaR95 Frozen  Viol={viol95_frozen.sum()} ({viol95_frozen.mean()*100:.1f}%)"),
    (axes[1,2], walk_dates, walk_ret, var95_wf, viol95_wf,
     f"VaR95 Walk-Fwd  Viol={viol95_wf.sum()} ({viol95_wf.mean()*100:.1f}%)"),
]:
    ax.plot(dates, ret, color='#555', lw=0.5, alpha=0.8, label='Return')
    ax.plot(dates, var, color='#1565C0', lw=0.9, label='VaR(95%)')
    ax.fill_between(dates, var, ret.min()*1.1, alpha=0.06, color='#1565C0')
    vm = np.asarray(viol) == 1
    ax.scatter(np.array(dates)[vm], ret[vm], s=12, color='#D32F2F',
               zorder=5, label='Violation')
    ax.axhline(0, color='k', lw=0.4, ls='--', alpha=0.4)
    ax.set_title(label, fontsize=9); ax.set_ylabel('Log Return')
    ax.legend(loc='lower left', fontsize=7); ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if 'Train' in label else 1))

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_master_overview.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig1_master_overview.png")


# ── Fig 2: RMSE / MAE / Corr comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Volatility Forecast Metrics: Train | Frozen | Walk-Forward',
             fontsize=12, fontweight='bold')

labels3 = ['Train\n(2006-2021)', 'Frozen\n(A-ver)', 'Walk-Fwd\n(B-ver)']
colors3  = ['#1565C0', '#E53935', '#F57C00']

for ax, name, vals in [
    (axes[0], 'RMSE', [rmse_tr,    rmse_frozen, rmse_wf]),
    (axes[1], 'MAE',  [mae_tr,     mae_frozen,  mae_wf]),
    (axes[2], 'Corr', [corr_tr,    corr_frozen, corr_wf]),
]:
    bars = ax.bar(labels3, vals, color=colors3, edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v + max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(name, fontsize=11); ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max(vals)*1.22)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_vol_metrics.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig2_vol_metrics.png")


# ── Fig 3: VaR violation rate comparison bar ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle('VaR(95%) Violation Rate — All Models', fontsize=12, fontweight='bold')

models_bar  = ['GK Realized\n(Benchmark)',
               'GARCH\n(Baseline)',
               'LSTM Frozen\n(A-ver)',
               'LSTM Walk-Fwd\n(B-ver)']
vrs_bar     = [k95_gkr['viol_rate'], k95_g['viol_rate'],
               k95_frz['viol_rate'], k95_wf['viol_rate']]
colors_bar  = ['#2E7D32', '#FF8F00', '#E53935', '#7B1FA2']

bars = ax.bar(models_bar, [v*100 for v in vrs_bar],
              color=colors_bar, edgecolor='white', width=0.5)
ax.axhline(5.0, color='navy', ls='--', lw=1.5, label='5% target')
for bar, v in zip(bars, vrs_bar):
    ax.text(bar.get_x()+bar.get_width()/2, v*100+0.2,
            f'{v*100:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Violation Rate (%)'); ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, max(vrs_bar)*100*1.25)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_var_violation.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig3_var_violation.png")


# ── Fig 4: Annual violation rate (Frozen vs Walk-Fwd) ────────────────────────
ann_frz = (pd.DataFrame({'year': walk_dates.year, 'viol': viol95_frozen})
           .groupby('year').agg(vr=('viol','mean')).reset_index())
ann_wf2 = (pd.DataFrame({'year': walk_dates.year, 'viol': viol95_wf})
           .groupby('year').agg(vr=('viol','mean')).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Annual VaR(95%) Violation Rate: Frozen vs Walk-Forward (2022-2025)',
             fontsize=12, fontweight='bold')
for ax, ann, title in [(axes[0], ann_frz, 'Frozen Model (A-ver)'),
                        (axes[1], ann_wf2, 'Walk-Forward (B-ver)')]:
    yvals = ann['vr'].values
    cols  = ['#D32F2F' if v>0.05 else '#2E7D32' for v in yvals]
    bars  = ax.bar(ann['year'].astype(str), yvals, color=cols, edgecolor='white')
    ax.axhline(0.05, color='navy', ls='--', lw=1.5, label='5% target')
    for bar, v in zip(bars, yvals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v*100:.1f}%',
                ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=11); ax.set_ylabel('Violation Rate')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_annual_violation.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig4_annual_violation.png")


# ── Fig 5: Training loss curve ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.history['loss'],     color='#1565C0', lw=1.5, label='Train Loss')
ax.plot(history.history['val_loss'], color='#E53935', lw=1.5, ls='--', label='Val Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Huber Loss (scaled)')
ax.set_title('Base Model Training Loss (EarlyStopping + ReduceLROnPlateau)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig5_training_loss.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig5_training_loss.png")


# ── Fig 6: Rolling 60-day violation rate (Walk-Forward) ──────────────────────
fig, ax = plt.subplots(figsize=(13, 4.5))
df_roll = pd.DataFrame({'viol_wf': viol95_wf, 'viol_frz': viol95_frozen},
                        index=walk_dates)
ax.plot(walk_dates, df_roll['viol_wf'].rolling(60).mean(),
        color='#7B1FA2', lw=1.5, label='Walk-Forward (B-ver)')
ax.plot(walk_dates, df_roll['viol_frz'].rolling(60).mean(),
        color='#E53935', lw=1.5, ls='--', label='Frozen (A-ver)')
ax.axhline(0.05, color='navy', ls='--', lw=1.2, label='5% target')
ax.set_ylabel('Rolling 60d Violation Rate')
ax.set_title('Rolling 60-Day Violation Rate: Walk-Forward vs Frozen (2022-2025)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30); ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig6_rolling_violation.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig6_rolling_violation.png")


# ── Fig 7: UC Coverage with Wilson CI ────────────────────────────────────────
def wilson_ci(x: int, n: int, z: float = 1.96):
    if n <= 0: return np.nan, np.nan
    p = x / n; denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    half   = (z * np.sqrt((p*(1-p) + z**2/(4*n))/n)) / denom
    return (center-half, center+half)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('UC Coverage Check (Observed Violation Rate + 95% CI)',
             fontsize=12, fontweight='bold')

for ax, label, pairs in [
    (axes[0], 'Train (2006-2021)',
     [("VaR95", ALPHA_95, viol95_tr), ("VaR99", ALPHA_99, viol99_tr)]),
    (axes[1], 'Frozen Test (2022-2025)',
     [("VaR95", ALPHA_95, viol95_frozen), ("VaR99", ALPHA_99, viol99_frozen)]),
    (axes[2], 'Walk-Fwd Test (2022-2025)',
     [("VaR95", ALPHA_95, viol95_wf), ("VaR99", ALPHA_99, viol99_wf)]),
]:
    xs, alphas, rates, ci_lo, ci_hi, ns, xvs = [], [], [], [], [], [], []
    for i, (lvl, alpha, viol) in enumerate(pairs):
        n2 = len(viol); x2 = int(viol.sum())
        p_hat = x2/n2
        lo, hi = wilson_ci(x2, n2)
        xs.append(i); alphas.append(alpha)
        rates.append(p_hat); ci_lo.append(p_hat-lo); ci_hi.append(hi-p_hat)
        ns.append(n2); xvs.append(x2)
    ax.errorbar(xs, rates, yerr=[ci_lo, ci_hi], fmt='o', capsize=5, ms=8,
                label='Observed (95% CI)', color='#1565C0')
    ax.plot(xs, alphas, 's--', color='#E53935', ms=8, label='Theoretical α')
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{pairs[j][0]}\n(n={ns[j]},x={xvs[j]})" for j in range(len(pairs))])
    ax.set_ylabel('Violation Rate'); ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig7_uc_coverage.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig7_uc_coverage.png")


# ── Fig 8: Hit timeline ───────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Hit Timeline: VaR(95%) Violations', fontsize=12, fontweight='bold')

for ax, dates, viol, label in [
    (axes[0], dates_tr,  viol95_tr,     'Train VaR95'),
    (axes[1], walk_dates, viol95_frozen, 'Frozen VaR95'),
    (axes[2], walk_dates, viol95_wf,     'Walk-Fwd VaR95'),
    (axes[3], walk_dates, viol95_wf,     'Walk-Fwd VaR99 (reference)'),
]:
    viol_arr = np.asarray(viol)
    hit_dates = np.array(dates)[viol_arr==1]
    if len(hit_dates) > 0:
        ax.eventplot(hit_dates, lineoffsets=1, linelengths=0.8, color='#D32F2F')
    ax.set_yticks([1]); ax.set_yticklabels([label], fontsize=8)
    ax.grid(True, axis='x', alpha=0.3)
    ax.set_title(f'{label}  (n={viol_arr.sum()})', fontsize=9)

axes[-1].set_xlabel('Date')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig8_hit_timeline.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig8_hit_timeline.png")


# ── Fig 9: Prediction diagnosis (variance suppression) ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Prediction Diagnosis: Variance Suppression & Bias',
             fontsize=12, fontweight='bold')

# Scatter: actual vs predicted
ax = axes[0]
ax.scatter(true_vol_te, pred_frozen, s=5, alpha=0.3, color='#E53935', label='Frozen')
ax.scatter(walk_vol, walk_pred, s=5, alpha=0.3, color='#7B1FA2', label='Walk-Fwd')
lim = max(true_vol_te.max(), pred_frozen.max(), walk_pred.max())*1.05
ax.plot([0,lim],[0,lim], 'k-', lw=1.5, label='45° (perfect)')
ax.set_xlabel('Actual gk_vol'); ax.set_ylabel('Predicted')
ax.set_title(f'Scatter (Test)\nFrozen Corr={corr_frozen:.3f}, WF Corr={corr_wf:.3f}', fontsize=9)
ax.legend(fontsize=8); ax.grid(alpha=0.25)

# Residual by quintile
ax = axes[1]
q = pd.qcut(true_vol_te, 5, labels=['Q1','Q2','Q3','Q4','Q5'])
bias_frz = [(pred_frozen[q==qn]-true_vol_te[q==qn]).mean() for qn in ['Q1','Q2','Q3','Q4','Q5']]
bias_wf  = [(walk_pred[q==qn]  -walk_vol[q==qn]).mean()    for qn in ['Q1','Q2','Q3','Q4','Q5']]
x5 = np.arange(5)
ax.bar(x5-0.2, bias_frz, 0.35, color='#E53935', label='Frozen', alpha=0.8)
ax.bar(x5+0.2, bias_wf,  0.35, color='#7B1FA2', label='Walk-Fwd', alpha=0.8)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_xticks(x5); ax.set_xticklabels(['Q1\n(low)','Q2','Q3','Q4','Q5\n(high)'])
ax.set_ylabel('Mean Bias (Pred - True)'); ax.set_title('Bias by Vol Quintile\n(negative=underestimate)', fontsize=9)
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Std ratio
ax = axes[2]
std_ratios = [pred_tr.std()/true_vol_tr.std(),
              pred_frozen.std()/true_vol_te.std(),
              walk_pred.std()/walk_vol.std()]
labels_std = ['Train', 'Frozen', 'Walk-Fwd']
bars = ax.bar(labels_std, std_ratios, color=['#1565C0','#E53935','#7B1FA2'],
              edgecolor='white', width=0.4)
ax.axhline(1.0, color='black', lw=1.5, ls='--', label='Perfect = 1.0')
for bar, v in zip(bars, std_ratios):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Pred Std / True Std'); ax.set_title('Variance Suppression\n(<1 = model compresses range)', fontsize=9)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 1.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig9_diagnosis.png"), dpi=200, bbox_inches='tight')
plt.close(); print("✓ Saved fig9_diagnosis.png")


# =============================================================================
# 10.  最終摘要列印
# =============================================================================

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print("\n[Volatility Forecast]")
print(f"  Base Model Train:  RMSE={rmse_tr:.6f}  MAE={mae_tr:.6f}  Corr={corr_tr:.4f}")
print(f"  Frozen A-ver Test: RMSE={rmse_frozen:.6f}  MAE={mae_frozen:.6f}  Corr={corr_frozen:.4f}")
print(f"  Walk-Fwd B-ver Test: RMSE={rmse_wf:.6f}  MAE={mae_wf:.6f}  Corr={corr_wf:.4f}")
print(f"  WF vs Frozen: RMSE {(rmse_wf-rmse_frozen)/rmse_frozen*100:+.1f}%  MAE {(mae_wf-mae_frozen)/mae_frozen*100:+.1f}%")

print("\n[VaR(95%) Backtest]")
rows_print = [
    ("GK Realized (benchmark)", k95_gkr, c95_gkr),
    ("GARCH (baseline)",        k95_g,   c95_g),
    ("LSTM Frozen A-ver",       k95_frz, c95_frz),
    ("LSTM Walk-Fwd B-ver",     k95_wf,  c95_wf),
    ("LSTM Base Train",         k95_tr,  c95_tr),
]
for name, k, c in rows_print:
    print(f"  {name:<28s}: {k['viol_rate']*100:.2f}%  "
          f"UC={'PASS' if k['p_uc']>=0.05 else 'FAIL'}(p={k['p_uc']:.4f})  "
          f"CC={'PASS' if c['p_cc']>=0.05 else 'FAIL'}(p={c['p_cc']:.4f})")

print(f"\n✓ All outputs saved to: {OUTPUT_DIR}/")
print("\nOutput files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    print(f"  {f:<45s} {os.path.getsize(fpath)//1024:5d} KB")

[Frozen A-ver ] RMSE=0.004236  MAE=0.002762  Corr=0.7201
✓ Base model saved.

STAGE 2: Walk-Forward Warm Update (2022-2025)
  Warm LR=0.0001, Warm Epochs=1, Warm Batch=1
✓ Warm model weights copied from base model.
Walk-forward window: 877 days
  Walk-forward:  200/877 days done
  Walk-forward:  400/877 days done
  Walk-forward:  600/877 days done
  Walk-forward:  800/877 days done
  Walk-forward:  877/877 days done

[Walk-Forward B-ver] RMSE=0.004245  MAE=0.002841  Corr=0.7166
[Frozen    A-ver   ] RMSE=0.004236  MAE=0.002762  Corr=0.7201
RMSE change: +0.2%  MAE change: +2.9%
✓ Warm model saved.

VaR(95%) Backtest Results
  [Base Model Train 2006-2021]  N=4025, Viol=277 (6.88%)  UC p=0.0000 ✗  CC p=0.0000 ✗  IND p=0.1610  n00/01/10/11=3495/252/252/25
  [Frozen  A-ver  Test 2022-2025]  N=877, Viol=74 (8.44%)  UC p=0.0000 ✗  CC p=0.0001 ✗  IND p=0.9121  n00/01/10/11=734/68/68/6
  [Walk-Fwd B-ver Test 2022-2025]  N=877, Viol=69 (7.87%)  UC p=0.0003 ✗  CC p=0.0014 ✗  IND p=0.7953  n00/01/1